In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce

In [ ]:
%sql
SHOW SCHEMAS IN workspace

Ingesting The data from csv

In [ ]:
retail_df = spark.read.format('csv')\
    .option('header', 'true')\
    .option('inferSchema', 'true')\
    .load('/Volumes/workspace/ecommerce/raw_data/online_retail_II.csv')


In [ ]:
retail_df.printSchema()
display(retail_df.limit(10))

display('Total number of records', retail_df.count())

In [ ]:
from pyspark.sql.functions import *

retail_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in retail_df.columns]).display()

In [ ]:
retail_df.printSchema()
retail_df = retail_df.withColumnRenamed("Customer ID","Customer_ID")


Bronze layer ingestion


In [ ]:

retail_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.bronze_retail")


Bronze Layer validation

In [ ]:
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.ecommerce.bronze_retail")

# 1. Total records
print("Total Records:", bronze_df.count())

# 2. Total columns
print("Total Columns:", len(bronze_df.columns))

# 3. Null count for every column
null_counts = bronze_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
])

display(null_counts)

# 4. Schema
bronze_df.printSchema()

# 5. Sample records
display(bronze_df.limit(10))

In [ ]:
from pyspark.sql.functions import *

#-------------------------------------------------------------CHECKING FOR THE PERCENTAGE OF NULL VALUES------------------------------------------------------
total_records = bronze_df.count()

null_percent_calculation = null_counts.select([col(c)/total_records * 100  for c in null_counts.columns])
display(null_percent_calculation)
        
#--------------------------------------------------------------------------------CHECKING FOR THE NEGATIVE VALUES AND THERE COUNT------------------------------------
negative_values = bronze_df.select([sum(when(col("Quantity") < 0,1).otherwise(0)).alias("Negative Values_Quantity")])
display(negative_values)

negative_values = bronze_df.select([sum(when(col("Price") < 0,1).otherwise(0)).alias("Negative Values_Price")])
display(negative_values)

In [ ]:
#Verifying the negative quantity data
display(bronze_df.filter(col("Quantity") < 0))

In [ ]:
#Finding the No of duplicate data
duplicate_df = bronze_df.distinct().count()

total_records = bronze_df.count()

print(f"Total Distinct Records: {duplicate_df}")
print(f"Total Records: {total_records}")
print("Total Duplicate Records:",total_records - duplicate_df)

